In [0]:
from pyspark.sql.functions import col, when, current_timestamp, coalesce, round, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print(" Silver schema ready")

In [0]:
# First clean invoice line items from source
line_items_clean = (
    spark.table("azure_blob_storage.src_invoice_line_items")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("discount", coalesce(col("discount"), lit(0.0)))
    .withColumn("quantity", when(col("quantity") < 0, 0).otherwise(col("quantity")))
    .withColumn("unit_price", when(col("unit_price") < 0, 0).otherwise(col("unit_price")))
    .withColumn("line_total", round(col("quantity") * col("unit_price") * (1 - col("discount")), 2))
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["invoice_line_id"])
)

# Join with products and invoices
invoice_line_items = (
    line_items_clean.alias("ili")
    .join(
        spark.table("silver.products").alias("prod"),
        col("ili.product_id") == col("prod.product_id"),
        "left"
    )
    .join(
        spark.table("silver.invoices").alias("inv"),
        col("ili.invoice_id") == col("inv.invoice_id"),
        "left"
    )
    .select(
        col("ili.invoice_line_id"),
        col("ili.invoice_id"),
        coalesce(col("inv.customer_id"), lit("none")).alias("customer_id"),
        coalesce(col("inv.customer_name"), lit("none")).alias("customer_name"),
        col("inv.invoice_date"),
        col("ili.product_id"),
        coalesce(col("prod.product_name"), lit("none")).alias("product_name"),
        coalesce(col("prod.category"), lit("none")).alias("category"),
        coalesce(col("prod.cost_price"), lit(0)).alias("cost_price"),
        col("ili.quantity"),
        col("ili.unit_price"),
        col("ili.discount"),
        col("ili.line_total"),
        col("ili.ingestion_ts")
    )
)

invoice_line_items.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.invoice_line_items")
print(f" invoice_line_items: {invoice_line_items.count()} records")

In [0]:
print("\nSample data (with product and invoice info):")
display(spark.table("silver.invoice_line_items").limit(5))